# CS604 Project - 05 Test-set submission (Phase 2)

Makes the Phase 2 zip (`test_team_XX.zip`) from a finished run. **Nothing is trained here**, so Run All is safe. Any Colab GPU is enough.

**Use the same model and settings as your posted validation entry** (course rule for Phase 2): the same run folder, the same `TTA` and the same test size `SIZE` (leave `SIZE = None` to use the run's training size).

Before you start: the run folder `My Drive/cs604_runs/<RUN>/` in YOUR Google Drive must hold that run's `config.yaml` and `final.pt` (for a run trained by someone else, copy those two files from their Drive into a folder of the same name in yours).

In [ ]:
REPO_URL = "https://github.com/notzdean/CS604-Computer-Vision.git"   # <- set once
RUN    = "final_5pct_trivialaug"   # 5% model (run 30); for full fine-tuning use "full_ft_llrd_288" (run 39)
TEAM   = "XX"                      # team number for the file name (test_team_XX.zip)
TTA    = "none"                    # must match the validation entry: "none", or "flip" if mirror averaging was used there
SIZE   = 288                       # test size in px. run 30 was trained at 224 px, its best test size is 288 (run 22).
                                   # None = the run's training size (288 for full_ft_llrd_288)


## 1. Code

In [ ]:
import getpass, os, subprocess
REPO = "/content/repo"

def git(*args, cwd=None):
    r = subprocess.run(["git", *args], capture_output=True, text=True, cwd=cwd)
    return r.returncode, (r.stdout + r.stderr).strip()

def with_token(url, token):                      # the token is only used for this command, never stored in the repo
    return url.replace("https://", f"https://x-access-token:{token}@")

token = os.environ.get("GITHUB_TOKEN", "")
if not os.path.isdir(f"{REPO}/.git"):
    code, out = git("clone", REPO_URL, REPO)     # works if the repo is public
    if code != 0:
        print("Clone failed (private repo?). Paste a GitHub token with read access to this repo.")
        token = getpass.getpass("GitHub token: ")
        code, out = git("clone", with_token(REPO_URL, token), REPO)
        if code == 0:
            git("remote", "set-url", "origin", REPO_URL, cwd=REPO)   # do not keep the token in .git/config
    assert code == 0, out.replace(token, "***") if token else out
else:
    code, out = git("pull", with_token(REPO_URL, token) if token else "origin", cwd=REPO)
    print(out.replace(token, "***") if token else out)
print("repo ready at", REPO)
!pip install -q -r /content/repo/requirements.txt
# Colab's old torchao (0.10) breaks peft's LoRA import; we do not use torchao, so remove it
!pip uninstall -y -q torchao

## 2. Test data only (about 9 GB, 5 to 10 min; skipped if already there)
The training set is not needed for predictions, so only the small label lists and the test images are downloaded.

In [ ]:
%%bash
set -e
set -o pipefail
D=/content/cs604-course-data
U=https://huggingface.co/datasets/doem1997/cs604-course-data/resolve/main
if [ -f $D/.test_ready ]; then echo "Test data already prepared - skipping download."; exit 0; fi
mkdir -p $D && cd $D
curl -sSfL $U/train.txt -o train.txt
curl -sSfL $U/val.txt -o val.txt
curl -sSfL $U/test.txt -o test.txt
curl -sSfL $U/test.tar.gz | tar -xz
touch .test_ready
echo "test images listed: $(wc -l < test.txt)"
df -h /content
ls $D


## 3. Google Drive and the run folder

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
OUT = "/content/drive/MyDrive/cs604_runs"
run_dir = f"{OUT}/{RUN}"
assert RUN, "set RUN in the first cell"
missing = [f for f in ("config.yaml", "final.pt") if not os.path.exists(f"{run_dir}/{f}")]
assert not missing, f"not found in {run_dir}: {missing}. Upload them to My Drive/cs604_runs/{RUN}/ (a new upload can take a minute to appear), then run this cell again."
%cd /content/repo
print("using", run_dir, "->", sorted(os.listdir(run_dir)))

## 4. Predict the test images and write the zip
Writes `test_team_XX.txt`, `test_team_XX.zip` and `test_top5.npz` into the run folder and checks that every test image appears exactly once. Upload the zip to CodaBench Phase 2 (**only 2 submissions in total**). You may rename the zip (for example `test_team_01_full_ft_llrd_288.zip`), but not the `.txt` inside it.

In [ ]:
size_arg = f"--size {SIZE}" if SIZE else ""
print("run:", RUN, "| tta:", TTA, "| size:", SIZE or "training size")
!python -m common.submit $OUT/$RUN --team $TEAM --split test --tta $TTA $size_arg

## 5. Quick check before uploading

In [ ]:
import zipfile
txt = f"{OUT}/{RUN}/test_team_{TEAM}.txt"
lines = open(txt).read().splitlines()
n_test = len(open("/content/cs604-course-data/test.txt").read().splitlines())
print(f"{len(lines)} predictions for {n_test} test images ->", "OK" if len(lines) == n_test else "MISMATCH, do not upload")
print("first lines:", *lines[:3], sep="\n  ")
print("zip holds:", zipfile.ZipFile(txt[:-4] + ".zip").namelist())
